# Phase 5 - Feature Scaling (Normalization)

This notebook implements Phase 5 of the Sector Rotation Regime Discovery project. 
It loads the finalized 42-feature baseline (36 sector features, 5 NIFTY 50 features, 1 VIX feature) and applies scaling logic. 

**Why scaling is required:**
Regime discovery relies on unsupervised clustering (e.g., HMM/GMM), which uses multi-dimensional distance metrics (like Mahalanobis or Euclidean distance). Because our features have vastly different scales (e.g., 200-day trend ratio vs. 20-day percentage returns), larger-scale features would mathematically dominate the clustering algorithm. Scaling normalizes the features so each contributes proportionally to the state discovery.

**Why RobustScaler is the primary choice:**
The EDA identified that our volatility and return features contain heavily skewed distributions and extreme outliers (representing genuine market stress). `StandardScaler` calculates mean and standard deviation, which are disproportionately inflated by these extreme values, crushing the variance of "normal" market periods. `RobustScaler` uses the Median and IQR (Interquartile Range) instead. This centers and scales the middle 50% of the data consistently across all sectors, preserving the massive magnitude of market crashes and rallies as uncompressed signals for the HMM.

**Why StandardScaler is used for comparison:**
We also produce a `StandardScaler` dataset to serve as a sensitivity benchmark in case the massive uncompressed tails from `RobustScaler` cause numerical instability (singular covariance matrices) during the GMM fitting process.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

from src.normalization import (
    load_master_feature_dataset, 
    temporal_train_test_split, 
    fit_transform_scaler,
    get_42_features
)

data_dir = project_root / 'data'

## 1. Extracting the 42 Features

We load the completed Phase 3 engineered files without modifying the raw data. We merge the exact 42 approved features.

In [2]:
master_df = load_master_feature_dataset(data_dir)
approved_features = get_42_features()

print(f"Loaded {len(master_df.columns)} columns.")
assert len(master_df.columns) == 42, "Error: Feature count is not exactly 42."
assert set(master_df.columns) == set(approved_features), "Error: Feature names do not match approved list."
print("Master dataset successfully loaded.")

Loaded 42 columns.
Master dataset successfully loaded.


## 2. Temporal Splitting Policy

**To prevent data leakage:** The scaling parameters (median/IQR or mean/variance) must be learned *exclusively* from historical training data. If we fitted the scaler on the full dataset, future events (like the 2020 COVID-19 crash) would corrupt the scaled values of historical 2019 data. 
For this initial pass, we define `2024-12-31` as the training cutoff.

In [3]:
train_end_date = '2024-12-31'
train_df, test_df = temporal_train_test_split(master_df, train_end_date)
print(f"Training Set: {train_df.index[0].date()} to {train_df.index[-1].date()} ({len(train_df)} rows)")
print(f"Testing Set:  {test_df.index[0].date()} to {test_df.index[-1].date()} ({len(test_df)} rows)")

Training Set: 2019-01-02 to 2024-12-31 (1464 rows)
Testing Set:  2025-01-02 to 2026-09-18 (422 rows)


## 3. Scaling Execution
We fit on the training data and transform both the training and test data using the fitted scaler. Initial rolling-window NaNs are preserved natively by scikit-learn scalers.

In [4]:
# RobustScaler (Primary)
train_robust, test_robust, robust_scaler = fit_transform_scaler(train_df, test_df, 'robust')

# StandardScaler (Sensitivity)
train_standard, test_standard, standard_scaler = fit_transform_scaler(train_df, test_df, 'standard')

## 4. Validation Steps
We validate that the scaling preserved the chronological order, missing value boundaries, feature order, and finite nature of the data.

In [5]:
def validate_scaled_data(original_df, scaled_train, scaled_test):
    scaled_full = pd.concat([scaled_train, scaled_test]).sort_index()
    
    # 1. Feature count & column order
    assert list(original_df.columns) == list(scaled_full.columns), "Column order mismatch."
    
    # 2. Date alignment
    assert original_df.index.equals(scaled_full.index), "Date index mismatch."
    
    # 3. Missing Values (NaN preservation)
    assert original_df.isna().sum().equals(scaled_full.isna().sum()), "Missing value count changed after scaling!"
    
    # 4. Finite values (check that scaling didn't produce inf)
    non_nans = scaled_full.dropna().values
    assert np.isfinite(non_nans).all(), "Infinite values detected in scaled data."
    
    print("✅ Validation passed. Extreme observations preserved. No artificial imputation detected.")

print("Validating Robust Scaler output:")
validate_scaled_data(master_df, train_robust, test_robust)

print("Validating Standard Scaler output:")
validate_scaled_data(master_df, train_standard, test_standard)

Validating Robust Scaler output:
✅ Validation passed. Extreme observations preserved. No artificial imputation detected.
Validating Standard Scaler output:
✅ Validation passed. Extreme observations preserved. No artificial imputation detected.


## 5. Save Outputs
We combine the train and test scaled DataFrames back together and save them to separate directories in `data/processed/normalized/`.

In [6]:
from src.normalization import scale_and_save_master_datasets

robust_full, standard_full = scale_and_save_master_datasets(data_dir, train_end_date)
print(f"Robust dataset saved to: data/processed/normalized/robust/master_features_robust.csv")
print(f"Standard dataset saved to: data/processed/normalized/standard/master_features_standard.csv")

Robust dataset saved to: data/processed/normalized/robust/master_features_robust.csv
Standard dataset saved to: data/processed/normalized/standard/master_features_standard.csv
